In [ ]:
#!/usr/bin/env python3
"""
Monitor RET-tramlijnen 3 en 5 bij de haltes Leuvehaven & Wilhelminaplein.

‣ Bronnen: v0.ovapi.nl (KV8 realtime JSON)
‣ Externe dependency: requests  →  pip install requests
"""

import math
import sys
from datetime import datetime
from time import sleep
from urllib.parse import quote_plus

import requests

# ────────────── Config ─────────────────────────────────────────
LINES       = (3, 5)                  # meerdere lijnen mogelijk
DIRECTIONS  = (1, 2)                  # 1 = ene richting, 2 = terug
BASE_URL    = "http://v0.ovapi.nl/line/RET_{line}_{dir}"
HEADERS     = {"User-Agent": "RET35StopMonitor/1.0 (contact: you@example.com)"}
TIMEOUT     = 10                      # seconden
MAX_DIST_M  = 550                     # drempel ‘bijna/aanwezig’
REFRESH_S   = 30                      # polling-interval (s); sla op 0 voor éénmalig

# Haltes: decimale WGS-84-coördinaten
STOPS = {
    "LEU": (51.913351, 4.482089),     # Leuvehaven
    "WIL": (51.908007, 4.491337),     # Wilhelminaplein
}
# ───────────────────────────────────────────────────────────────


# ───── Helpers ────────────────────────────────────────────────
def haversine_m(lat1, lon1, lat2, lon2):
    """Retourneer afstand in meter tussen twee WGS-coördinaten."""
    R = 6_371_000
    f1, f2 = math.radians(lat1), math.radians(lat2)
    df = f2 - f1
    dl = math.radians(lon2 - lon1)
    a = math.sin(df / 2) ** 2 + math.cos(f1) * math.cos(f2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def maps_link(lat, lon):
    return "https://www.google.com/maps/search/?api=1&query=" + quote_plus(f"{lat},{lon}")


def fetch_actuals(line: int, direction: int) -> dict:
    url = BASE_URL.format(line=line, dir=direction)
    try:
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        r.raise_for_status()
        return r.json().get(f"RET_{line}_{direction}", {}).get("Actuals", {})
    except requests.RequestException as exc:
        print(f"⚠️  API-fout (lijn {line}, dir {direction}): {exc}")
        return {}


def coords(rec: dict):
    lat = rec.get("Latitude") or rec.get("latitude")
    lon = rec.get("Longitude") or rec.get("longitude")
    return lat, lon
# ───────────────────────────────────────────────────────────────


def check_trams():
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    found = False

    for line in LINES:
        for direction in DIRECTIONS:
            for jid, rec in fetch_actuals(line, direction).items():
                lat, lon = coords(rec)
                if lat is None or lon is None:
                    continue    # geen GPS-fix

                for code, (s_lat, s_lon) in STOPS.items():
                    dist = haversine_m(lat, lon, s_lat, s_lon)
                    if dist <= MAX_DIST_M:
                        found = True
                        print(f"[{ts}] lijn {line}  rit {jid}  (dir {direction})  "
                              f"➜ {code}  ({int(dist)} m)")
                        print(f"      ↗ {maps_link(lat, lon)}\n")
                        break     # per tram één halte tonen

    if not found:
        print(f"[{ts}] Geen tram van lijn {LINES} binnen {MAX_DIST_M} m "
              f"van LEU of WIL.")


def main():
    if REFRESH_S <= 0:          # éénmalige check
        check_trams()
        return

    try:
        while True:
            check_trams()
            sleep(REFRESH_S)
    except KeyboardInterrupt:
        sys.exit("\nAfgebroken door gebruiker.")


if __name__ == "__main__":
    main()


[2025-05-20 16:29:43] Geen tram van lijn (3, 5) binnen 550 m van LEU of WIL.
